In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data_folder = '../data/'

fichiers_csv = sorted([
    f for f in os.listdir(data_folder)
    if f.endswith('.csv') and f != 'processed_dataset.csv'
])
print(f'{len(fichiers_csv)} fichiers trouvés')

In [ ]:
def calculer_features(df, ride_id):
    d = df.copy()
    d['ride_id'] = ride_id

    delta_km = d['km'].diff().fillna(0)
    delta_secs = d['secs'].diff().fillna(1).replace(0, 1)
    d['vitesse_kmh'] = (delta_km / delta_secs * 3600).clip(0, 120)

    delta_alt = d['alt'].diff().fillna(0)
    delta_m = delta_km * 1000
    d['pente_pct'] = np.where(delta_m > 0, (delta_alt / delta_m) * 100, 0)
    d['pente_pct'] = d['pente_pct'].clip(-30, 30)

    d['acceleration'] = d['vitesse_kmh'].diff().fillna(0).clip(-10, 10)
    d['delta_hr'] = d['hr'].diff().fillna(0).clip(-10, 10)

    d['vitesse_moy_5s'] = d['vitesse_kmh'].rolling(window=5, min_periods=1).mean()
    d['hr_moy_5s'] = d['hr'].rolling(window=5, min_periods=1).mean()
    d['cad_moy_5s'] = d['cad'].rolling(window=5, min_periods=1).mean()
    d['pente_moy_5s'] = d['pente_pct'].rolling(window=5, min_periods=1).mean()

    return d

print('OK')

In [ ]:
liste_df_enrichis = []

for i, fichier in enumerate(fichiers_csv):
    chemin = os.path.join(data_folder, fichier)
    df_temp = pd.read_csv(chemin)
    ride_id = fichier.replace('.csv', '')
    liste_df_enrichis.append(calculer_features(df_temp, ride_id))

    if (i + 1) % 20 == 0:
        print(f'{i+1}/{len(fichiers_csv)}')

df_final = pd.concat(liste_df_enrichis, ignore_index=True)
print(f'Dataset final : {df_final.shape[0]} lignes, {df_final.shape[1]} colonnes')
df_final.head()

In [ ]:
print(df_final.isnull().sum())
df_final = df_final.dropna()
print(f'Lignes restantes : {len(df_final)}')

In [ ]:
df_actif = df_final[df_final['power'] > 0]
echantillon = df_actif.sample(5000, random_state=42)

plt.figure(figsize=(10, 6))
plt.scatter(echantillon['pente_pct'], echantillon['power'], alpha=0.2, color='darkorange', s=10)
plt.title('Pente vs Puissance')
plt.xlabel('Pente (%)')
plt.ylabel('Puissance (W)')
plt.tight_layout()
plt.savefig('../plots/pente_vs_power.png')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(echantillon['vitesse_kmh'], echantillon['power'], alpha=0.2, color='steelblue', s=10)
plt.title('Vitesse vs Puissance')
plt.xlabel('Vitesse (km/h)')
plt.ylabel('Puissance (W)')
plt.tight_layout()
plt.savefig('../plots/vitesse_vs_power.png')
plt.show()

In [ ]:
chemin_sortie = '../data/processed_dataset.csv'
df_final.to_csv(chemin_sortie, index=False)
print(f'Sauvegardé : {chemin_sortie}')
print(df_final.shape)
print(list(df_final.columns))